# Practice guide 04_01: static malware detection from PE headers and API fingerprints

**Goal.** Build the static detector yourself: the evaluation metrics of an imbalanced scanner (AUC, average
precision, precision/recall at an operating point), the benign-downsampling experiment, a two-stage
hierarchy over 7 malware families, and the malleable-field preview that class 04_03 exploits. Every `TODO`
cell must be completed; the `check` cells tell you whether you got it right.

Read `guide_04_01.pdf` first. Time: about 2 hours. Cells marked **(given)** are complete.

In [3]:
import sys
from pathlib import Path

import numpy as np
from scipy import sparse as sp
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

LIB = next(p for p in (Path("../../notebooks/03-malware"), Path("../../../notebooks/03-malware")) if p.exists())
sys.path.insert(0, str(LIB))  # student / solution folder
import malwlib as mw

## Part A. The PE-header data (given)

19,611 files x 77 numeric header fields, 74.4% malware — the *reverse* of a real scanner queue, which we
exploit in part C.

In [4]:
X, y, cols = mw.load_pe()
print(f"PE data: {X.shape[0]:,} files x {X.shape[1]} fields, malware {y.mean():.2%}")
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

PE data: 19,611 files x 77 fields, malware 74.44%


**Question A1.** Why does `stratify=y` matter here less than in a 1%-malware problem — and when would it
matter at all?

## Part B. Ranking metrics from scratch

A scanner outputs a score; the analyst picks a threshold. Implement, from the sorted scores alone:

* `auc_score(y, s)`: the probability that a random malware file scores above a random benign one (handle
  ties with 0.5);
* `average_precision(y, s)`: precision summed at the rank of each positive, divided by the number of
  positives;
* `pr_at_threshold(y, s, t)`: precision and recall at threshold `t`.

No scikit-learn in your implementation — the check cells compare with it.

In [5]:
def rank_matrix(y, s):
    """Sort by score descending; return the sorted labels and the scores."""
    o = np.argsort(s)[::-1]
    return y[o], s[o]


def auc_score(y, s):
    pos, neg = np.sort(s[y == 1]), np.sort(s[y == 0])
    gt = np.searchsorted(neg, pos, side="right").sum()
    eq = np.searchsorted(neg, pos, side="left").sum() - gt
    return (gt + 0.5 * eq) / (len(pos) * len(neg))


def average_precision(y, s):
    yy, _ = rank_matrix(y, s)
    c = np.cumsum(yy)
    ranks = np.arange(1, len(yy) + 1)
    prec = c / ranks
    return float(prec[yy == 1].mean())


def pr_at_threshold(y, s, t):
    pred = s >= t
    tp = int((pred & (y == 1)).sum())
    p = tp / int(pred.sum()) if pred.sum() else 0.0
    r = tp / int((y == 1).sum())
    return p, r

In [6]:
# check: a perfect and a random score, then the scikit-learn twins
from sklearn.metrics import average_precision_score, roc_auc_score

rng = np.random.default_rng(0)
yp = np.concatenate([np.ones(50), np.zeros(50)])
assert auc_score(yp, yp) == 1.0 and abs(auc_score(yp, rng.random(100)) - 0.5) < 0.15
s = rng.random(100) * np.where(yp, 1, 0.6)
assert abs(auc_score(yp, s) - roc_auc_score(yp, s)) < 1e-12
assert abs(average_precision(yp, s) - average_precision_score(yp, s)) < 1e-12
print(f"AUC {auc_score(yp, s):.3f}  AP {average_precision(yp, s):.3f}  metrics OK")

AUC 0.689  AP 0.768  metrics OK


**Question B1.** A scanner flags a file when its score is above the threshold. With 74.4% malware in the
test set, what is the precision of a scanner that flags *every* file? And with the realistic 1% base rate?
Which quantity does the base rate enter directly?

## Part C. The detector and the imbalance experiment

Train the random forest (given), then downsample the benign *training* files to the fractions 1.0, 0.2, 0.05,
0.01 and measure, for each, the AUC, the AP and the precision at fixed recall 0.95. Fill the table.

In [7]:
rf_full = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr, ytr)
p_full = rf_full.predict_proba(Xte)[:, 1]
print("full model:", mw.eval_bin(yte, p_full, "RF PE baseline: "))

RF PE baseline: AUC 0.9992  AUPRC 0.9997  F1@.5 0.9941
full model: {'auc': 0.9991839858312653, 'auprc': 0.9996960283681545, 'f1': 0.9941296928327645}


In [8]:
idx_b = np.where(ytr == 0)[0]
rowsC = {}
rng = np.random.default_rng(0)
idx_m = np.where(ytr == 1)[0]
for frac in [1.0, 0.2, 0.05, 0.01]:
    keep_b = rng.choice(idx_b, size=int(len(idx_b) * frac), replace=False)
    rows = np.concatenate([idx_m, keep_b])
    rf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr[rows], ytr[rows])
    p = rf.predict_proba(Xte)[:, 1]
    o = np.argsort(p)[::-1]
    yy, ss = yte[o], p[o]
    c = np.cumsum(yy)
    k = int(np.argmax(c >= 0.95 * yy.sum()))
    t = ss[k]
    pr, _ = pr_at_threshold(yte, p, t)
    rowsC[frac] = (auc_score(yte, p), average_precision(yte, p), pr)
for frac, (a, ap_, pr) in rowsC.items():
    print(f"benign frac {frac:5.2f}  AUC {a:.4f}  AP {ap_:.4f}  precision@recall.95 {pr:.4f}")

benign frac  1.00  AUC 0.9992  AP 0.9997  precision@recall.95 0.9986
benign frac  0.20  AUC 0.9950  AP 0.9971  precision@recall.95 0.9949
benign frac  0.05  AUC 0.9903  AP 0.9917  precision@recall.95 0.9912
benign frac  0.01  AUC 0.9869  AP 0.9904  precision@recall.95 0.9896


**Question C1.** Which column degrades as the benign fraction drops, and which barely moves? What does that
mean for a vendor who trains on a 1%-malware queue and advertises an AUC?

## Part D. API-function fingerprints: flat vs two-stage

The Windows API dataset counts 21,918 API functions per file over 7 classes. Loading it eagerly would need
tens of GB; `malwlib.load_api_sparse` reads 100-row UInt8 slices and keeps a stratified 20,000-file sample
(~6 MB CSR, peak RSS under 0.8 GB). **This cell takes a few minutes the first time; the demo cache makes it
instant afterwards.**

In [9]:
CACHE = mw.ROOT / ".cache_api_sub"
if CACHE.with_suffix(".npz").exists():
    M, yf = sp.load_npz(CACHE.with_suffix(".npz")), np.load(CACHE.with_suffix(".npy"))
else:
    M, yf = mw.load_api_sparse()
    sp.save_npz(CACHE.with_suffix(".npz"), M)
    np.save(CACHE.with_suffix(".npy"), yf)
print(f"API matrix: {M.shape[0]:,} x {M.shape[1]:,}, density {M.nnz / (M.shape[0] * M.shape[1]):.4f}")

itr, ite = train_test_split(np.arange(len(yf)), test_size=0.25, stratify=yf, random_state=0)

API matrix: 20,000 x 21,918, density 0.0032


Train the flat 7-class forest (given) and the two-stage pipeline: stage 1 binary benign-vs-malware, stage 2
family only on the files stage 1 calls malicious. Compare macro-F1 end to end.

In [10]:
flat = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1).fit(M[itr], yf[itr])
pf = flat.predict(M[ite])
from sklearn.metrics import classification_report, f1_score

print(classification_report(yf[ite], pf, digits=3, zero_division=0))
flat_macro = f1_score(yf[ite], pf, average="macro")

              precision    recall  f1-score   support

           0      0.942     0.412     0.573       318
           1      0.904     0.886     0.895       851
           2      0.988     0.982     0.985       787
           3      0.946     0.461     0.620       840
           4      0.892     0.748     0.813       860
           5      0.340     0.985     0.506       717
           6      0.600     0.038     0.072       627

    accuracy                          0.684      5000
   macro avg      0.802     0.645     0.638      5000
weighted avg      0.806     0.684     0.669      5000



In [11]:
yb = (yf > 0).astype(int)
hier = {}
rf1 = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1).fit(M[itr], yb[itr])
pred1 = rf1.predict(M[ite])
mal_tr = itr[yb[itr] == 1]
rf2 = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1).fit(M[mal_tr], yf[mal_tr])
sel = pred1 > 0
fam2 = rf2.predict(M[ite[sel]])
end = np.zeros(len(yf[ite]), dtype=int)
end[sel] = fam2
stage2_rows = yf[ite][sel] > 0
hier = (
    float((pred1[yb[ite] == 1] == 1).mean()),
    f1_score(yf[ite][sel][stage2_rows], fam2[stage2_rows], average="macro"),
    f1_score(yf[ite], end, average="macro"),
)
print(f"flat macro-F1 {flat_macro:.4f}")
print(f"hier: stage-1 recall {hier[0]:.4f}, stage-2 macro-F1 {hier[1]:.4f}, end-to-end macro-F1 {hier[2]:.4f}")

flat macro-F1 0.6378
hier: stage-1 recall 0.9991, stage-2 macro-F1 0.6563, end-to-end macro-F1 0.6350


**Question D1.** The hierarchy wins (or loses) only a little macro-F1. Why does the *recall of stage 1* cap
the whole pipeline, and why can stage 2 never recover a file stage 1 missed?

## Part E. What the model looks at — and the preview of 04_03

Print the 8 most important header fields. `malwlib.MALLEABLE` lists the fields an attacker can rewrite
without breaking the program: count how many of the top-20 fields are malleable.

In [12]:
imp = rf_full.feature_importances_
top = np.argsort(imp)[::-1]
for i in top[:8]:
    print(f"{cols[i]:34s} {imp[i]:.3f}")
n_mal = sum(cols[i] in mw.MALLEABLE for i in top[:20])
print(f"malleable fields among the top 20: {n_mal}")

MinorOperatingSystemVersion        0.087
MajorSubsystemVersion              0.085
SizeOfStackReserve                 0.081
MajorOperatingSystemVersion        0.080
MajorLinkerVersion                 0.075
Subsystem                          0.061
TimeDateStamp                      0.061
MinorImageVersion                  0.045
malleable fields among the top 20: 10


**Question E1.** Class 04_03 rewrites exactly these fields to the benign median and the detector folds. Two
defences come to mind that do **not** need retraining — name them and say why each is unsatisfying.

## Part G. A safe replicator — what the detector cannot even see

`malwlib.replicate` writes the source of a minimal self-copying program, generation by generation,
into a scratch directory. It carries no payload (it only writes the next generation and stops at a
hard cap), is never executed as a subprocess, and touches nothing outside `workdir`. Build 5
generations and read generation 0.

In [13]:
import tempfile

workdir_g = Path(tempfile.mkdtemp(prefix="guide_04_01_replicator_"))
paths_g = mw.replicate(workdir_g, cap=5)
print(f"{len(paths_g)} generations written to {workdir_g}")
print(paths_g[0].read_text())

6 generations written to C:\Users\parli\AppData\Local\Temp\guide_04_01_replicator_c43sf4cm
#!/usr/bin/env python3
"""Generation 0: a benign replicator. Copies this file to the next generation, then stops."""
import pathlib

GEN = 0
CAP = 5
SOURCE = '#!/usr/bin/env python3\n"""Generation {gen}: a benign replicator. Copies this file to the next generation, then stops."""\nimport pathlib\n\nGEN = {gen}\nCAP = {cap}\nSOURCE = {source!r}\n\ndef main():\n    if GEN >= CAP:\n        print(f"generation {{GEN}}: cap reached, stopping")\n        return\n    here = pathlib.Path(__file__)\n    next_path = here.with_name(f"replicator_gen{{GEN + 1}}.py")\n    next_path.write_text(SOURCE.format(gen=GEN + 1, cap=CAP, source=SOURCE))\n    print(f"generation {{GEN}}: wrote {{next_path.name}}")\n\nif __name__ == "__main__":\n    main()\n'

def main():
    if GEN >= CAP:
        print(f"generation {GEN}: cap reached, stopping")
        return
    here = pathlib.Path(__file__)
    next_path = here.with_nam

**Question G1.** Could you build a 77-column row for this file the way `load_pe` does for the PE
dataset? Could a static detector trained on PE headers, or on API-import counts, return *any* score
for it? What specifically is missing — the schema, or the training data?

In [14]:
import shutil

shutil.rmtree(workdir_g)  # clean up: this demo is confined to its own temp directory

# Reference solution

The solved notebook, with the answers to the questions and the numbers of a full run, is `solution/guide_04_01.ipynb`.